In [16]:
# Setup

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction


In [17]:
# Imports

import pandas as pd

from src.data_sources.ports import load_port_files, process_ports
from src.data_sources.weather import load_weather_files, process_weather
from src.features.weather_features import (
    prepare_port_reference_for_weather,
    create_weather_history_features,
    merge_weather_features,
)

In [18]:
# Paths

internal_features_path = PROJECT_ROOT / "data" / "interim" / "internal_features.parquet"
raw_ports_dir = PROJECT_ROOT / "data" / "raw" / "ports"
raw_weather_dir = PROJECT_ROOT / "data" / "raw" / "weather"

df_internal = pd.read_parquet(internal_features_path)

print(df_internal.shape)
df_internal.head()

(141895, 69)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,...,arrival_is_weekend,arrival_shift,arrival_season,avg_wait_prev_20_calls_port,avg_operation_prev_20_calls_port,std_wait_prev_20_calls_port,arrival_date,arrivals_same_day_port,arrivals_prev_day_port,arrivals_prev_7d_avg_port
0,561202021,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,1,night,summer,NaN,NaN,NaN,2022-01-02,1,NaN,NaN
1,4172022,BR052001,TERMINAL FLUVIAL DE JURUTI,9304241,<NA>,GOOD HOPE MAX,Carga,BR052,JURUTI,BRAMW001,...,0,evening,summer,NaN,NaN,NaN,2022-01-06,1,1.0,NaN
2,7012022,BR052001,TERMINAL FLUVIAL DE JURUTI,9628922,<NA>,AMBERJACK,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001,...,1,evening,summer,NaN,NaN,NaN,2022-01-08,1,1.0,NaN
3,12292022,BR052001,TERMINAL FLUVIAL DE JURUTI,9473339,<NA>,JURUTI,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,0,morning,summer,NaN,NaN,NaN,2022-01-13,1,1.0,1.0
4,19472022,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BR052,JURUTI,BR052,...,0,night,summer,NaN,NaN,NaN,2022-01-17,1,1.0,1.0


In [19]:
# Load and process ports data

df_ports_raw = load_port_files(raw_ports_dir)
df_ports = process_ports(df_ports_raw, coord_decimals=5)
df_ports = prepare_port_reference_for_weather(df_ports)

print(df_ports.shape)
df_ports.head()

(125, 9)


,port,port_name,city,state,region,latitude,longitude,latitude_r,longitude_r
0,BR052001,TERMINAL FLUVIAL DE JURUTI,JURITI,PA,NORTE,-2.173782,-56.105980,-2.17378,-56.10598
1,BR147001,TERMINAL PORTUÁRIO FLUVIAL - COPESUL,TRIUNFO,RS,SUL,-29.948610,-51.623888,-29.94861,-51.62389
2,BR252001,TERMINAL AQUAVIARIO DE GUAMARE - TRANSPETRO,GUAMARÉ,RN,NORDESTE,-4.873611,-36.374166,-4.87361,-36.37417
3,BR252002,TERM AQUAVIARIO DE GUAMARE - 3R OPERAÇÕES MARI...,GUAMARÉ,RN,NORDESTE,-5.106387,-36.317918,-5.10639,-36.31792
4,BR302002,TUP NSR,NOVA SANTA RITA,RS,SUL,-29.931204,-51.282781,-29.93120,-51.28278


In [20]:
# Load and process weather data

df_weather_raw = load_weather_files(raw_weather_dir)
df_weather = process_weather(df_weather_raw, coord_decimals=5)

print(df_weather.shape)
df_weather.head()

(161117, 14)


,latitude,longitude,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,wind_gusts_10m_max,wind_direction_10m_dominant,latitude_r,longitude_r
0,-32.122222,-52.093333,2021-06-15,14.0,15.9,12.9,0.0,0.0,0.0,34.4,46.1,203,-32.12222,-52.09333
1,-32.122222,-52.093333,2021-06-16,12.5,13.3,11.8,0.0,0.0,0.0,33.6,46.1,203,-32.12222,-52.09333
2,-32.122222,-52.093333,2021-06-17,12.0,12.5,11.5,0.7,0.7,4.0,21.3,28.8,222,-32.12222,-52.09333
3,-32.122222,-52.093333,2021-06-18,12.7,14.1,11.7,0.1,0.1,1.0,16.9,25.9,140,-32.12222,-52.09333
4,-32.122222,-52.093333,2021-06-19,14.5,14.9,13.8,6.6,6.6,22.0,29.1,34.2,69,-32.12222,-52.09333


In [21]:
# Creating weather history

df_weather_feat = create_weather_history_features(df_weather)

print(df_weather_feat.shape)
df_weather_feat.head()

(161117, 29)


,latitude,longitude,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,...,rain_sum_prev_3d,precipitation_hours_prev_3d,temperature_2m_mean_prev_3d,wind_speed_10m_max_prev_3d,wind_gusts_10m_max_prev_3d,rain_sum_prev_7d,precipitation_hours_prev_7d,temperature_2m_mean_prev_7d,wind_speed_10m_max_prev_7d,wind_gusts_10m_max_prev_7d
0,-32.122222,-52.093333,2021-06-15,14.0,15.9,12.9,0.0,0.0,0.0,34.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-32.122222,-52.093333,2021-06-16,12.5,13.3,11.8,0.0,0.0,0.0,33.6,...,0.0,0.0,14.000000,34.4,46.1,NaN,NaN,NaN,NaN,NaN
2,-32.122222,-52.093333,2021-06-17,12.0,12.5,11.5,0.7,0.7,4.0,21.3,...,0.0,0.0,13.250000,34.4,46.1,NaN,NaN,NaN,NaN,NaN
3,-32.122222,-52.093333,2021-06-18,12.7,14.1,11.7,0.1,0.1,1.0,16.9,...,0.7,4.0,12.833333,34.4,46.1,0.7,4.0,12.833333,34.4,46.1
4,-32.122222,-52.093333,2021-06-19,14.5,14.9,13.8,6.6,6.6,22.0,29.1,...,0.8,5.0,12.400000,33.6,46.1,0.8,5.0,12.800000,34.4,46.1


In [22]:
# Merge to the analytical dataset

df_eda = merge_weather_features(
    df=df_internal,
    weather_df=df_weather_feat,
    ports_df=df_ports,
)

print(df_eda.shape)
df_eda.head()

(141895, 104)


,port_call_id,port,port_name_x,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,...,temperature_2m_mean_prev_3d,wind_speed_10m_max_prev_3d,wind_gusts_10m_max_prev_3d,rain_sum_prev_7d,precipitation_hours_prev_7d,temperature_2m_mean_prev_7d,wind_speed_10m_max_prev_7d,wind_gusts_10m_max_prev_7d,has_port_reference,has_weather_data
0,561202021,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,26.466667,11.5,26.6,142.2,89.0,26.914286,14.2,31.7,1,1
1,4172022,BR052001,TERMINAL FLUVIAL DE JURUTI,9304241,<NA>,GOOD HOPE MAX,Carga,BR052,JURUTI,BRAMW001,...,25.966667,8.9,23.4,159.9,111.0,26.242857,12.2,27.7,1,1
2,7012022,BR052001,TERMINAL FLUVIAL DE JURUTI,9628922,<NA>,AMBERJACK,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BRAMW001,...,26.233333,9.2,26.6,162.8,119.0,25.900000,12.2,27.7,1,1
3,12292022,BR052001,TERMINAL FLUVIAL DE JURUTI,9473339,<NA>,JURUTI,Carga,BRAMW001,TERMINAL DA ALUMAR - SÃO LUIZ - MA,BR052,...,26.700000,9.6,24.1,66.8,99.0,26.114286,10.6,26.6,1,1
4,19472022,BR052001,TERMINAL FLUVIAL DE JURUTI,9620310,<NA>,WHITE WHALE,Carga,BR052,JURUTI,BR052,...,27.400000,13.6,29.9,31.3,74.0,27.042857,13.6,29.9,1,1


In [23]:
df_eda["has_port_reference"].value_counts(dropna=False)

has_port_reference
1    141895
Name: count, dtype: Int64

In [24]:
df_eda["has_weather_data"].value_counts(dropna=False)

has_weather_data
1    141844
0        51
Name: count, dtype: Int64

In [25]:
# Analyze missingness in the new columns

weather_cols = [
    "city",
    "state",
    "region",
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "rain_sum",
    "precipitation_hours",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "wind_direction_10m_dominant",
    "rain_sum_prev_1d",
    "temperature_2m_mean_prev_3d",
    "wind_speed_10m_max_prev_3d",
    "rain_sum_prev_7d",
    "wind_gusts_10m_max_prev_7d",
]

df_eda[weather_cols].isna().mean().sort_values(ascending=False)

temperature_2m_mean_prev_3d    0.000381
rain_sum_prev_1d               0.000381
wind_speed_10m_max_prev_3d     0.000381
rain_sum_prev_7d               0.000381
wind_gusts_10m_max_prev_7d     0.000381
temperature_2m_min             0.000359
temperature_2m_max             0.000359
precipitation_sum              0.000359
temperature_2m_mean            0.000359
wind_speed_10m_max             0.000359
wind_direction_10m_dominant    0.000359
precipitation_hours            0.000359
rain_sum                       0.000359
wind_gusts_10m_max             0.000359
city                           0.000000
region                         0.000000
state                          0.000000
dtype: float64

In [26]:
df_eda[df_eda["has_weather_data"] == False]

,port_call_id,port,port_name_x,imo,vessel_id,vessel_name,operation_type,source_port,source_port_name,destination_port,...,temperature_2m_mean_prev_3d,wind_speed_10m_max_prev_3d,wind_gusts_10m_max_prev_3d,rain_sum_prev_7d,precipitation_hours_prev_7d,temperature_2m_mean_prev_7d,wind_speed_10m_max_prev_7d,wind_gusts_10m_max_prev_7d,has_port_reference,has_weather_data
15168,105612013,BRFOR,FORTALEZA (MUCURIPE),8716241,<NA>,WS ITAQUI,Solicitação de certificado,BRFOR,FORTALEZA (MUCURIPE),<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
15169,13742020,BRFOR,FORTALEZA (MUCURIPE),<NA>,182-002087-8,ARAXÁ,Solicitação de certificado,BRNAT,NATAL,BRNAT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
15170,469412020,BRFOR,FORTALEZA (MUCURIPE),9668697,3813896897,SAAM PAITER,Solicitação de certificado,BRPEC,PECEM,BRBEL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
15171,478392020,BRFOR,FORTALEZA (MUCURIPE),8716227,1210104814,ENGENHEIRO MASCARENHAS,Solicitação de certificado,BRFOR,FORTALEZA (MUCURIPE),<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
15172,188182021,BRFOR,FORTALEZA (MUCURIPE),<NA>,3813876454,WS VIRGO,Abastecimento (Bunker),BRFOR,FORTALEZA (MUCURIPE),BRFOR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
17137,142952021,BRIBB,IMBITUBA,9440617,4019900671,DRACO,Desembarque/Embarque de Passageiros,BRITJ,ITAJAI,BRSSZ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
26889,244412019,BRITJ,ITAJAI,<NA>,381-387544-0,ITABIRA,Solicitação de certificado,BRVIX,VITÓRIA,BRSUA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
26890,332022019,BRITJ,ITAJAI,9188130,<NA>,BRAM NATAL,Solicitação de certificado,BRRIO,RIO DE JANEIRO,BRNTR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
26891,13962021,BRITJ,ITAJAÍ,9664299,443-048077-0,BRAM BRASILIA,Solicitação de certificado,BRRIO,RIO DE JANEIRO,BRRIO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
26892,44622021,BRITJ,ITAJAÍ,9696802,<NA>,STARNAV AQUARIUS,Solicitação de certificado,BRRIO,RIO DE JANEIRO,BRRIO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0


In [ ]:
# Manual check

cols = [
    "port",
    "city",
    "state",
    "region",
    "arrival_port_ts",
    "arrival_date",
    "latitude_r",
    "longitude_r",
    "rain_sum",
    "rain_sum_prev_3d",
    "wind_speed_10m_max",
    "wind_gusts_10m_max_prev_7d",
]

df_eda[cols].head(20)

,port,city,state,region,arrival_port_ts,arrival_date,latitude_r,longitude_r,rain_sum,rain_sum_prev_3d,wind_speed_10m_max,wind_gusts_10m_max_prev_7d
0,BR052001,JURITI,PA,NORTE,2022-01-02 00:24:00,2022-01-02,-2.17378,-56.10598,8.2,106.3,12.2,31.7
1,BR052001,JURITI,PA,NORTE,2022-01-06 23:54:00,2022-01-06,-2.17378,-56.10598,13.3,45.4,7.9,27.7
2,BR052001,JURITI,PA,NORTE,2022-01-08 21:06:00,2022-01-08,-2.17378,-56.10598,23.7,29.6,10.6,27.7
3,BR052001,JURITI,PA,NORTE,2022-01-13 11:50:00,2022-01-13,-2.17378,-56.10598,4.2,12.8,12.4,26.6
4,BR052001,JURITI,PA,NORTE,2022-01-17 00:18:00,2022-01-17,-2.17378,-56.10598,21.4,14.3,8.7,29.9
5,BR052001,JURITI,PA,NORTE,2022-01-20 12:00:00,2022-01-20,-2.17378,-56.10598,1.2,57.6,15.8,32.0
6,BR052001,JURITI,PA,NORTE,2022-01-25 11:30:00,2022-01-25,-2.17378,-56.10598,6.1,47.1,11.2,38.5
7,BR052001,JURITI,PA,NORTE,2022-01-26 17:36:00,2022-01-26,-2.17378,-56.10598,21.9,42.8,10.2,38.5
8,BR052001,JURITI,PA,NORTE,2022-01-30 02:00:00,2022-01-30,-2.17378,-56.10598,3.6,65.3,12.7,34.2
9,BR052001,JURITI,PA,NORTE,2022-01-31 14:00:00,2022-01-31,-2.17378,-56.10598,3.3,24.6,9.6,34.2


In [ ]:
# Saving

interim_dir = PROJECT_ROOT / "data" / "interim"
processed_dir = PROJECT_ROOT / "data" / "processed"

interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

ports_clean_path = interim_dir / "ports_clean.parquet"
weather_clean_path = interim_dir / "weather_clean.parquet"
weather_features_path = interim_dir / "weather_features.parquet"
eda_base_path = processed_dir / "eda_base.parquet"

df_ports.to_parquet(ports_clean_path, index=False)
df_weather_feat.to_parquet(weather_clean_path, index=False)
df_eda.to_parquet(weather_features_path, index=False)
df_eda.to_parquet(eda_base_path, index=False)

print(f"Ports clean saved to: {ports_clean_path}")
print(f"Weather clean saved to: {weather_clean_path}")
print(f"Weather features saved to: {weather_features_path}")
print(f"EDA base saved to: {eda_base_path}")